- https://thinkingmachines.ai/blog/on-policy-distillation/
- https://huggingface.co/spaces/HuggingFaceH4/on-policy-distillation
- https://zhuanlan.zhihu.com/p/1992696535025214593
- https://zhuanlan.zhihu.com/p/2025536259628569667
- glm-5
    - on-policy cross-stage distillation

### core questions

- kd vs. opd
- sft, rl, opd
- token-level vs. Sequence-level estimator
    - bias, variance
    - 二者的关键差异是是否让当前 token 的更新承担未来轨迹（future-reward）的影响。
    - seq level: future coupled
- sampled-token vs. full vocabulary vs. Top-K 支持集

### KD -> OPD

- 离线 SFT 或离线蒸馏学习的是已有样本上的行为,样本前缀未必来自当前学生模 型。学生一旦在推理时走到训练分布之外,后面的错误会被前缀漂移放大。

- 经典 KD 目标: hard label CE（标准的 supervised learning，Groud truth supervision） 加 soft KL（teacher soft label supervision），分类场景中,经典 KD 常写成 hard-label loss 与 teacher soft-target loss 的加权和:
$$L_{KD}=(1-\alpha)L_{\text{hard}}+\alpha\tau^2 L_{\text{soft}}$$
- $L_{\text{hard}}=-\log q_{\theta,\tau}(y|s)$
- $L_{\text{soft}}=D_{KL}(p_{T,\tau}(\cdot|s)\|q_{\theta,\tau}(\cdot|s))=\sum_{a}p_{T,\tau}(a|s)[\log p_{T,\tau}(a|s)-\log q_{\theta,\tau}(a|s)]$
- 因为 $p_{T,\tau}$ 固定，优化 student 时，
    - $\nabla_\theta D_{KL}(p_{T,\tau}\|q_{\theta,\tau})=-\sum_ap_{T,\tau}(a|s)\nabla_\theta q_{\theta,\tau}(a|s)$

#### why fkl

机器学习概率建模的根基是最大似然估计（MLE），即寻找一组参数 $\theta$，使得这组模型生成这批真实数据的概率最大：
$$
\theta_{\text{MLE}} = \arg\max_{\theta} \mathbb{E}_{x \sim P_{\text{data}}} [\log Q_{\theta}(x)]
$$

在KD中：在经典 KD 里，soft loss 的原始出发点通常是：teacher 给出一个 soft label 分布 $p_{T,\tau}(\cdot|s)$（teacher soft label），student 在这个“软标签数据分布”下做最大似然。也就是假设类别 $a$ 按 teacher 分布产生：
- Teacher 模型就是那个产生数据的“自然界/真实分布” $P_{\text{data}}$（或者说 $P_{T}$）。
- Student 模型就是那个等待训练的参数化模型 $Q_{\theta}$。

Student 的训练目标完全沿袭了 MLE 的思想：在给定输入 $s$（状态/样本）的情况下，让 Student 输出的预测 $q_{\theta}(a|s)$ 在 Teacher 的概率分布 $p_{T}(a|s)$ 下的对数似然最大。

$$
\begin{split}
\min_{\theta} L(\theta) &= - \mathbb{E}_{a \sim p_{T}(\cdot|s)} [\log q_{\theta}(a|s)] = - \sum_{a} p_{T}(a|s) \log q_{\theta}(a|s)\\
&= \sum_{a} p_{T}(a|s) \log p_{T}(a|s) - \sum_{a} p_{T}(a|s) \log q_{\theta}(a|s) - \sum_{a} p_{T}(a|s) \log p_{T}(a|s)\\
&= \sum_{a} p_{T}(a|s) \log \frac{p_{T}(a|s)}{q_{\theta}(a|s)} - \sum_{a} p_{T}(a|s) \log p_{T}(a|s)
\end{split}
$$

$$
\nabla_{\theta} L_{\text{MLE}} \equiv \nabla_{\theta} H(p_{T}, q_{\theta}) \equiv \nabla_{\theta} D_{\mathrm{KL}}(p_{T} \parallel q_{\theta})
$$

- $H(P,Q)=H(P)+D_{\mathrm{KL}}(P\|Q)$
    - $-\sum_x P(x)\log Q(x)=-\sum_x P(x)\log P(x)+\sum_x P(x)\log \frac{P(x)}{Q(x)}$

### OPD 与 OPSD

> 学生自己写,老师逐 token 批改
- On Policy Distillation of Language Models: Learning from Self-Generated Mistakes
    - ICLR 2024, https://arxiv.org/abs/2306.13649
    - Generalized Knowledge Distillation (GKD)
    - 学生模型不只在固定数据集或 teacher 生成的序列上学习,而是可以采样学生自己生成的序列,再让  teacher 在这些学生前缀上给 token 级概率。
- 设 teacher 为 $\pi_T$，student 为 $\pi_\theta$。给定输入 $x$ 和某条输出前缀 $y_{\lt t}$，两者都能给出下一个 token 的分布:
    - $\pi_T(\cdot|x,y_{\lt t}), \pi_\theta(\cdot|x,y_{\lt t})$
- GKD 先定义一个逐 token 的差异度量 $d(\cdot,\cdot)$，例如 forward KL、reverse KL 或 JSD。对一整条序列 $y_{1\cdots T}$，可以把 teacher 和 student 在每个前缀后的差异平均起来:
    - $\Delta(x, y) = \frac{1}{T} \sum_{t=1}^T d(\pi_T(\cdot|x, y_{<t}), \pi_\theta(\cdot|x, y_{<t})).$
- $\Delta(x, y)$：teacher 对 student 这条回答的逐位置批改总分。
- GKD 的 on-policy 版本把 $y$ 改成 student 自己采样出来的回答:
    - $\mathcal{L}_{\text{OPD}} = \mathbb{E}_{x \sim D, y \sim \pi_\theta(\cdot|x)} [\Delta(x, y)].$

#### OPSD

- Self-Distilled Reasoner: On-Policy Self-Distillation for Large  Language Models
- 它保留 OPD 的  “student 自己生成回答”这一点,但不再要求一个外部 teacher。teacher 和 student 是同一个基础模 型,只是看到的信息不同:
    - student policy:只看到问题 $x$,这和测试时一致。
    - teacher policy:看到问题 $x$,还看到 privileged information $r$,例如参考解、正确答案或 verified  reasoning trace。
    - rollout:仍然由 student 采样, $y\sim\pi_\theta^S(\cdot|x)$
$$
\mathcal{L}_{\text{OPSD}} = \mathbb{E}_{(x,r)\sim D, y\sim\pi_\theta^S(\cdot|x)} \left[ \frac{1}{T} \sum_{t=1}^T d\left(\pi_\theta^T(\cdot|x, r, y_{<t}), \pi_\theta^S(\cdot|x, y_{<t})\right) \right].
$$
- OPSD 论文还提出 per-token pointwise divergence clipping。原因很具体:某些风格 token 的 KL 可  能很高,但它们不一定是数学推理的关键 token。如果不裁剪,训练会把大量更新用在“wait“”therefore” 这类风格差异上,而不是算式、数字、关系词等真正影响答案的 token 上。
- STaR 一类方法通常是生成 reasoning trace,筛掉错的,再对保留下来的硬 token 序列做 SFT。

### OPD 介于 SFT 和 RL 之间

- reverse-KL 视角,可以把 OPD 的一个局部目标写成:
    - $\mathcal{L}_{\text{OPD}} = \mathbb{E}_{s \sim d_{\pi_s}} [\text{KL}(\pi_s(\cdot|s)||\pi_T(\cdot|s))] = \mathbb{E}_{s \sim d_{\pi_s}, a \sim \pi_s} [\log \pi_s(a|s) - \log \pi_T(a|s)].$
    - $\log \pi_s(a|s) - \log \pi_T(a|s)$：类似把 RL 的 advantage 权重换成 student/teacher logratio。
- teacher 决定“往哪里学”,student 的 on-policy rollouts 决定“在哪些状态上学”。这就是为什  么 OPD 可能从一个已经遗忘的 SFT teacher 身上蒸馏出较少遗忘的 student:student 没有被  teacher 的外部训练数据分布直接牵引,而是在自己的状态分布上接收 teacher 的局部建议。
- OPD 给了每个 token 更密集的分布信号,但 teacher logits 本身可能带偏差。RLVR 的 outcome reward 很稀疏,却可能更接近任务真实成功;teacher distillation 很密,却可能把风格、推理  模板、无关 token 也当成要学习的东西。

### reuse RL framework

> 最小化 reverse KL 等价于一种 RL 最大化问题（reverse KL 的梯度就是 policy gradient）

$$
\begin{split}
\nabla_{\theta}\mathcal{L}_{RL} &= - \mathbb{E}_{y \sim \pi_{\theta}} \left[ \nabla_{\theta}\log\pi_{\theta}(y) \cdot A(y) \right]\\
\mathcal{L}_{OPD}(\theta) &= D_{KL}(\pi_{\theta} || \pi_{E}) = \sum_{y} \pi_{\theta}(y) \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}
\end{split}
$$
- $\nabla_{\theta} D_{KL}(\pi_{\theta} || \pi_{E}) = \sum_{y} \left( \nabla_{\theta}\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)} + \pi_{\theta}(y) \cdot \nabla_{\theta} \left[ \log \pi_{\theta}(y) - \log \pi_{E}(y) \right] \right)$
    - 右侧部分 $\pi_{\theta}(y) \cdot \nabla_{\theta}\log\pi_{\theta}(y) \rightarrow \sum_{y} \nabla_{\theta}\pi_{\theta}(y) = \nabla_{\theta} \left( \sum_{y} \pi_{\theta}(y) \right)=0$
- $\nabla_{\theta} D_{KL} = \sum_{y} \nabla_{\theta}\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}=\sum_{y} \pi_{\theta}(y) \nabla_{\theta}\log\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}= - \mathbb{E}_{y \sim \pi_{\theta}} \left[ \nabla_{\theta}\log\pi_{\theta}(y) \cdot \log \frac{\pi_{E}(y)}{\pi_{\theta}(y)} \right]$
- 定义：$A(y) = \log \frac{\pi_{E}(y)}{\pi_{\theta}(y)}$
    - $A_t = \text{sg}\left[\log\frac{\pi_{E_i}(y_t|x, y_{<t})}{\pi_\theta(y_t|x, y_{\le t})}\right]$

### background

- 离线蒸馏 (Off-Policy Distillation) —— 传统的“背参考答案”
    - 学生只见过正确答案。到了考场上，如果学生自己写错了一个字（比如把“太阳”写成了“太阴”），因为它从来没见过这种情况，不知道后面该怎么圆回来，可能会开始胡言乱语（这就是著名的“分布偏移”问题）。
- 在线策略蒸馏 (On-Policy Distillation) —— 高效的“模拟考精讲”
    - 学生自己做题（生成内容）。不管学生写得好还是烂，老师都在旁边看着。
    - 老师会针对学生写出的具体内容进行打分和纠正。
    - 学生写了“太阴”，老师马上指出：“这里错了，你应该写‘阳’，但既然你已经写了‘太’，下一个字最大概率是‘阳’”。
    - 学生不仅学到了正确答案，还学会了如何从自己的错误中恢复过来。这比单纯背答案效果好得多。